# Wire line S2 Connector: auto-label, review, train, export

Free Colab GPU, no API. **First:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells
top to bottom (Runtime → Run all).

1. Upload `frames.zip` from `training/s2/` (or the loose frames).
2. Grounding DINO draws boxes for this station's classes on every frame.
3. **Review every frame** (the team guide asks for every box to be checked) and drop the bad ones.
4. Train a small YOLO model (10–20 min).
5. Download **`s2-connector.onnx`** and send it to Claude (or put it in `app/models/`).

Classes, in this exact order (the order `/line` expects): `red_wire, white_wire, black_wire, connector_open, connector_closed`

In [ ]:
# 1. GPU check and install
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU! Runtime > Change runtime type > T4 GPU"
!pip -q install -U ultralytics onnx
import torch; print("GPU available:", torch.cuda.is_available())

In [ ]:
# 2. Upload frames.zip (training/s2/frames.zip on your laptop). Loose .jpg frames also work.
import os, zipfile, glob, shutil
from pathlib import Path

WORK = Path("/content/s2")
(WORK / "frames").mkdir(parents=True, exist_ok=True)
if not glob.glob(str(WORK / "frames" / "*.jpg")):
    from google.colab import files
    print("Choose training/s2/frames.zip ...")
    for name in files.upload():
        if zipfile.is_zipfile(name):
            zipfile.ZipFile(name).extractall(WORK)
        elif name.lower().endswith((".jpg", ".jpeg", ".png")):
            shutil.move(name, WORK / "frames" / name)
FRAMES = sorted(glob.glob(str(WORK / "frames" / "*.jpg")))
VIDEOS = sorted({Path(f).stem.rsplit("_", 1)[0] for f in FRAMES})
print(len(FRAMES), "frames from", len(VIDEOS), "videos:", VIDEOS)

In [ ]:
# 3a. Settings
STATION = "s2"
CLASSES = ["red_wire", "white_wire", "black_wire", "connector_open", "connector_closed"]          # do not reorder: /line reads them in this order
PROMPT = "red wire. white wire. black wire. orange lever connector."
ONE_PER_IMAGE = ["connector_open", "connector_closed"]   # classes that exist once per frame
BOX_THRESHOLD, TEXT_THRESHOLD = 0.30, 0.25
DINO_MODEL = "IDEA-Research/grounding-dino-base"
EXPORT_NAME = "s2-connector.onnx"

### 3a-2. When are the connector levers open?

Grounding DINO finds *the connector* but can't tell open from closed, so each video gets a list of frame ranges
(frame number = the last 4 digits of the frame name, 6 per second). `open` / `closed` set the class;
`skip` = the levers are being changed in the hand, those frames are left out.
Claude pre-filled these from the videos. In the review (step 4), **green** = open and **purple** = closed:
if a whole stretch has the wrong colour, fix the numbers here and run 3c again.
A new video must be added here too, or its connector frames are left out.

In [ ]:
STATES = {
 "IMG_0814": [
  [
   1,
   36,
   "closed"
  ],
  [
   37,
   75,
   "skip"
  ],
  [
   76,
   175,
   "open"
  ],
  [
   176,
   183,
   "skip"
  ],
  [
   184,
   9999,
   "closed"
  ]
 ],
 "IMG_0816": [
  [
   1,
   36,
   "closed"
  ],
  [
   37,
   72,
   "skip"
  ],
  [
   73,
   160,
   "open"
  ],
  [
   161,
   171,
   "skip"
  ],
  [
   172,
   9999,
   "closed"
  ]
 ]
}

def frame_state(stem):
    video, n = stem.rsplit("_", 1); n = int(n)
    for a, b, st in STATES.get(video, []):
        if a <= n <= b: return st
    return "skip"
for v, rs in STATES.items(): print(v, rs)

In [ ]:
# 3b. Turn Grounding DINO's boxes into labels (several wires per frame are allowed)
import numpy as np, cv2

def text_to_class(text):
    t = text.lower()
    if "cutter" in t or "plier" in t: return "cutter"
    if "ruler" in t: return "ruler"
    if "connector" in t or "lever" in t: return "connector?"   # open/closed comes from STATES
    if "wire" in t or "cable" in t:
        for c in ("red", "white", "black"):
            if c in t: return c + "_wire"
        return "wire?"                      # colour decided below
    return None

def red_share(img, box):
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = img[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0: return 0.0
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    red = ((h <= 10) | (h >= 165)) & (s > 90) & (v > 60)
    return float(red.mean())

def dark_share(img, box):
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = img[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0: return 0.0
    return float((cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)[..., 2] < 50).mean())

def wire_colour(img, box):
    # Fallback when the text gives no colour: red is easy to measure; dark vs light decides black vs white.
    if red_share(img, box) > 0.03: return "red_wire"
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = cv2.cvtColor(img[max(0, y1):y2, max(0, x1):x2], cv2.COLOR_BGR2HSV)
    v = crop[..., 2].reshape(-1)
    return "black_wire" if (v < 50).mean() > 0.05 else "white_wire"

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua else 0

def inside(a, b):
    # share of box a that lies inside box b
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    area = max(1e-6, (a[2]-a[0]) * (a[3]-a[1]))
    return max(0, x2 - x1) * max(0, y2 - y1) / area

def pick_boxes(img, boxes, scores, texts, state=None):
    H, W = img.shape[:2]
    cand = []
    for box, score, text in zip(boxes, scores, texts):
        x1, y1, x2, y2 = [float(v) for v in box]
        area = (x2 - x1) * (y2 - y1) / (W * H)
        if area > 0.6 or area < 0.0003:
            continue
        cls = text_to_class(text)
        if cls is None:
            continue
        if cls == "wire?":
            cls = wire_colour(img, box)
        elif cls == "connector?":
            if state not in ("open", "closed"): continue
            cls = "connector_" + state
        elif cls.endswith("_wire"):
            r = red_share(img, box)
            if cls == "red_wire" and r < 0.005 and score < 0.5: cls = wire_colour(img, box)
            elif cls != "red_wire" and r > 0.08: cls = "red_wire"     # clearly red: fix the text
        if cls not in CLASSES:
            continue
        bw, bh = x2 - x1, y2 - y1
        if cls == "ruler" and max(bw, bh) < 3 * min(bw, bh):
            continue                        # a ruler is long and thin; this is the whole cutting board
        if cls == "red_wire" and red_share(img, box) < 0.01:
            continue                        # nothing red in it
        if cls == "black_wire" and dark_share(img, box) < 0.03:
            continue                        # nothing black in it (e.g. the board's metal rail)
        cand.append([cls, float(score), [x1, y1, x2, y2]])
    cand.sort(key=lambda c: -c[1])
    keep = []
    cutters = [c[2] for c in cand if c[0] == "cutter"]
    conns = [c[2] for c in cand if c[0].startswith("connector")]
    for c in cand:                          # one box per object: drop a box that mostly repeats a better one
        if any(iou(c[2], k[2]) > 0.6 for k in keep):
            continue
        if c[0].endswith("_wire") and any(inside(c[2], k) > 0.5 for k in cutters):
            continue                        # the cutter's red handles are not a red wire
        if c[0].endswith("_wire") and any(inside(c[2], k) > 0.7 for k in conns):
            continue                        # the connector's orange levers are not a red wire
        if c[0].startswith("connector") and any(k[0].startswith("connector") for k in keep):
            continue                        # one connector per frame at this station
        if c[0] in ONE_PER_IMAGE and any(k[0] == c[0] for k in keep):
            continue
        keep.append(c)
    return [(c[0], c[2]) for c in keep]

def to_yolo_lines(objs, W, H):
    out = []
    for cls, (x1, y1, x2, y2) in objs:
        x1, x2 = max(0, x1), min(W, x2); y1, y2 = max(0, y1), min(H, y2)
        out.append(f"{CLASSES.index(cls)} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    return out

In [ ]:
# 3c. Auto-label every frame (a few minutes on a T4)
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(DINO_MODEL)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL).to(device).eval()

def detect(pil):
    inputs = processor(images=pil, text=PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        out = dino(**inputs)
    kw = dict(text_threshold=TEXT_THRESHOLD, target_sizes=[pil.size[::-1]])
    try:
        r = processor.post_process_grounded_object_detection(out, inputs.input_ids, threshold=BOX_THRESHOLD, **kw)[0]
    except TypeError:
        r = processor.post_process_grounded_object_detection(out, inputs.input_ids, box_threshold=BOX_THRESHOLD, **kw)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy(), list(r.get("text_labels", r.get("labels")))

LABELS = WORK / "labels_all"
LABELS.mkdir(exist_ok=True)
SKIPPED = []                                 # frames whose connector state is "skip"
has_states = "STATES" in globals()
counts = {c: 0 for c in CLASSES}
for f in tqdm(FRAMES):
    pil = Image.open(f).convert("RGB")
    img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    state = frame_state(Path(f).stem) if has_states else None
    if state == "skip":
        SKIPPED.append(Path(f).stem); continue
    objs = pick_boxes(img, *detect(pil), state=state)
    for c, _ in objs: counts[c] += 1
    (LABELS / (Path(f).stem + ".txt")).write_text("\n".join(to_yolo_lines(objs, *pil.size)))
print("Boxes per class:", counts)
if SKIPPED: print(len(SKIPPED), "frames skipped (levers changing):", SKIPPED[0], "...", SKIPPED[-1])

## 4. Review every frame

The pages below show **all** frames, 24 at a time, with their boxes:
**red** = red_wire, **white** = white_wire, **black (yellow outline)** = black_wire, **orange** = cutter, **blue** = ruler,
**green** = connector_open, **purple** = connector_closed. Skipped frames show "SKIP" and are left out anyway.

Write down the names of frames that are clearly wrong (a missing wire, a wrong colour, a box on a hand),
put them in `BAD` in cell 4b and run it. Those frames are left out of training. A few small misses are fine.
If most frames are wrong, change `BOX_THRESHOLD` in 3a (try 0.25 or 0.35) and run 3c again.

In [ ]:
# 4a. Show all labeled frames, one page at a time. Change PAGE and run again (PAGE = 0, 1, 2, ...)
import matplotlib.pyplot as plt
PAGE = 0
COLORS = {"red_wire": (230, 40, 40), "white_wire": (255, 255, 255), "black_wire": (255, 220, 0),
          "cutter": (255, 140, 0), "ruler": (0, 150, 255),
          "connector_open": (0, 200, 0), "connector_closed": (170, 0, 255)}

def draw(f):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    lab = LABELS / (Path(f).stem + ".txt")
    for line in (lab.read_text().splitlines() if lab.exists() else []):
        c, x, y, w, h = line.split(); name = CLASSES[int(c)]
        x, y, w, h = float(x)*W, float(y)*H, float(w)*W, float(h)*H
        cv2.rectangle(img, (int(x-w/2), int(y-h/2)), (int(x+w/2), int(y+h/2)), COLORS[name], max(2, W//250))
    return img

pages = (len(FRAMES) + 23) // 24
chunk = FRAMES[PAGE*24:(PAGE+1)*24]
print(f"Page {PAGE} of {pages - 1}")
fig, axes = plt.subplots(4, 6, figsize=(24, 18))
for ax in axes.flat: ax.axis("off")
for ax, f in zip(axes.flat, chunk):
    skip = Path(f).stem in set(globals().get("SKIPPED", []))
    ax.imshow(draw(f)); ax.set_title(Path(f).stem + ("  SKIP" if skip else ""), fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 4b. Frames to leave out (names as shown above), e.g. BAD = ["sample1-S1_0042", "sample2-S1_0107"]
BAD = []
USE = [f for f in FRAMES if Path(f).stem not in set(BAD) | set(globals().get("SKIPPED", []))]
print(len(USE), "frames kept,", len(FRAMES) - len(USE), "left out")

In [ ]:
# 5. Build the dataset. Split by video, never by frame: with 3+ videos one whole video is the check set;
#    with fewer, every 4th 2-second block of each video is (continuous stretches, so every step of the video
#    is in both sets, but neighbouring near-identical frames never end up on both sides).
import yaml, random
DS = WORK / "dataset"
shutil.rmtree(DS, ignore_errors=True)
by_video = {}
for f in USE: by_video.setdefault(Path(f).stem.rsplit("_", 1)[0], []).append(f)
train, val = [], []
if len(by_video) >= 3:
    hold = sorted(by_video)[-1]
    for v, fs in by_video.items(): (val if v == hold else train).extend(fs)
    print("Check set = video", hold)
else:
    for v, fs in by_video.items():
        for f in sorted(fs):
            block = int(Path(f).stem.rsplit("_", 1)[1]) // 12
            (val if block % 4 == 3 else train).append(f)
    print("Check set = every 4th 2-second block of each video")
for split, fs in (("train", train), ("val", val)):
    (DS / "images" / split).mkdir(parents=True); (DS / "labels" / split).mkdir(parents=True)
    for f in fs:
        shutil.copy(f, DS / "images" / split)
        lab = LABELS / (Path(f).stem + ".txt")
        (DS / "labels" / split / lab.name).write_text(lab.read_text() if lab.exists() else "")
(DS / "data.yaml").write_text(yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val",
                                             "names": {i: n for i, n in enumerate(CLASSES)}}))
print("train:", len(train), " check:", len(val))

In [ ]:
# 6. Train (10-20 min on a T4)
from ultralytics import YOLO
try:
    model = YOLO("yolo26n.pt")
except Exception:
    model = YOLO("yolo11n.pt")
model.train(
    data=str(DS / "data.yaml"), epochs=80, patience=25, imgsz=640, batch=16,
    project=str(WORK / "runs"), name=STATION, exist_ok=True,
    # Overhead camera: any rotation or flip is realistic; strong brightness and colour changes = robust to lighting.
    # hsv_h stays small so red, white and black keep their colour.
    degrees=180, flipud=0.5, fliplr=0.5, hsv_h=0.01, hsv_s=0.5, hsv_v=0.5, scale=0.4,
)
BEST = WORK / "runs" / STATION / "weights" / "best.pt"
print("Saved:", BEST)

In [ ]:
# 7. How good is it? mAP50 near 1.0 = very good, above 0.8 = fine to start.
best = YOLO(str(BEST))
m = best.val(data=str(DS / "data.yaml"), verbose=False)
print(f"mAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}")
for i, n in enumerate(CLASSES):
    ap = dict(zip(m.box.ap_class_index.tolist(), m.box.ap50.tolist())).get(i)
    print(f"  {n:11s} " + (f"mAP50 = {ap:.3f}" if ap is not None else "not in the check set"))
show = sorted(glob.glob(str(DS / "images" / "val" / "*.jpg")))[::max(1, len(val) // 6)][:6]
fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 6))
for ax, f in zip(np.atleast_1d(axes), show):
    ax.imshow(best.predict(f, conf=0.4, verbose=False)[0].plot()[:, :, ::-1]); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 8. Export to ONNX for /line (opset 12, 640 px, raw head) and download it, plus best.pt and the labeled data
onnx_path = best.export(format="onnx", opset=12, imgsz=640, simplify=False)
shutil.copy(onnx_path, f"/content/{EXPORT_NAME}")
shutil.copy(BEST, f"/content/{STATION}-best.pt")
shutil.make_archive(f"/content/{STATION}-dataset", "zip", DS)
print("Classes in order:", CLASSES)
from google.colab import files
files.download(f"/content/{EXPORT_NAME}")
files.download(f"/content/{STATION}-best.pt")